1- **Create Table on BigQuery**

In [3]:
from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# GOOGLE BIGQUERY AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

# =========================================
# CONFIGURATION
# =========================================

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Available_Visits"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

# =========================================
# INITIALIZE BIGQUERY CLIENT
# =========================================

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

# =========================================
# TABLE SCHEMA
# =========================================

schema = [

    bigquery.SchemaField(
        "Visit_Key",
        "STRING",
        mode="REQUIRED"
    ),

    bigquery.SchemaField(
        "Dr_Code",
        "STRING"
    ),

    bigquery.SchemaField(
        "Speciality",
        "STRING"
    ),

    bigquery.SchemaField(
        "Dr_Name",
        "STRING"
    ),

    bigquery.SchemaField(
        "Scheduled_Day",
        "STRING"
    ),

    bigquery.SchemaField(
        "Scheduled_Date",
        "DATE"
    ),

    bigquery.SchemaField(
        "Scheduled_Start_Time",
        "TIME"
    ),

    bigquery.SchemaField(
        "Scheduled_End_Time",
        "TIME"
    ),

    bigquery.SchemaField(
        "Shift_Name",
        "STRING"
    ),

    bigquery.SchemaField(
        "Capacity",
        "INTEGER"
    ),

    bigquery.SchemaField(
        "Remaining_Capacity",
        "INTEGER"
    ),

    bigquery.SchemaField(
        "Status",
        "STRING"
    )
]

# =========================================
# CREATE TABLE
# =========================================

table = bigquery.Table(
    TABLE_REF,
    schema=schema
)

table.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY,
    field="Scheduled_Date"
)

table.clustering_fields = [
    "Dr_Code",
    "Scheduled_Date",
    "Status"
]

table = client.create_table(
    table,
    exists_ok=True
)

print("=================================")
print("Table created successfully")
print(TABLE_REF)
print("=================================")

Table created successfully
depihealthnux.Depihealthnux_Main.Available_Visits


2- **Create Table on Postgres**

In [5]:
import sys
import psycopg2

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# CONNECT TO POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

# =========================================
# CREATE SEQUENCE
# =========================================

cursor.execute("""

CREATE SEQUENCE IF NOT EXISTS visit_no_seq;

""")

# =========================================
# CREATE TABLE
# =========================================

cursor.execute("""

CREATE TABLE IF NOT EXISTS available_visits (

    -- Auto Generated Primary Key
    visit_key TEXT PRIMARY KEY
    DEFAULT ('V_' || nextval('visit_no_seq')),

    -- Doctor Information
    dr_code TEXT,
    speciality TEXT,
    dr_name TEXT,

    -- Schedule Information
    scheduled_date DATE NOT NULL,
    scheduled_day TEXT NOT NULL,

    scheduled_start_time TIME NOT NULL,
    scheduled_end_time TIME NOT NULL,

    -- Generated in Python before insert
    shift_name TEXT NOT NULL,

    -- Capacity
    capacity INTEGER NOT NULL DEFAULT 0,

    remaining_capacity INTEGER NOT NULL DEFAULT 0,

    -- Status
    status TEXT,

    -- Validations
    CONSTRAINT chk_capacity
        CHECK (capacity >= 0),

    CONSTRAINT chk_remaining_capacity
        CHECK (
            remaining_capacity >= 0
            AND remaining_capacity <= capacity
        ),

    CONSTRAINT chk_visit_time
        CHECK (
            scheduled_end_time > scheduled_start_time
        ),

    -- Prevent duplicate slots
    CONSTRAINT uq_doctor_slot
        UNIQUE (
            dr_code,
            scheduled_date,
            scheduled_start_time
        )

);

""")

# =========================================
# INDEXES
# =========================================

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_available_visits_date
ON available_visits (scheduled_date);

""")

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_available_visits_doctor
ON available_visits (dr_code);

""")

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_available_visits_status
ON available_visits (status);

""")

conn.commit()

print("=================================")
print("PostgreSQL table created successfully")
print("Table: available_visits")
print("=================================")

# =========================================
# CLOSE CONNECTION
# =========================================

cursor.close()
conn.close()

PostgreSQL table created successfully
Table: available_visits
